<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 01
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Phân tích thị trường mỹ phẩm nội vs ngoại trên Tiki"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  TIỀN XỬ LÝ BỘ DỮ LIỆU TỪ TIKI
</div>

## **1. NHẬP DỮ LIỆU VÀ KIỂM TRA TỔNG QUAN**

In [17]:
import pandas as pd
import numpy as np
import re
import unicodedata

path = "../data" 
df = pd.read_csv(f"{path}/tiki_cosmetics_raw.csv")

print(f"Shape: {df.shape}")
print(f"\n--- Kiểu dữ liệu")
print(df.dtypes)
print(f"\n--- Tỉ lệ missing (%)")
missing = (df.isnull().mean() * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False))
print(f"\n--- Thống kê cơ bản")
print(df[["price","original_price","discount_rate","rating","review_count","sold_count"]].describe())

Shape: (8141, 21)

--- Kiểu dữ liệu
product_id               int64
name                    object
brand_id               float64
brand_name              object
seller_id                int64
seller_name             object
category                object
primary_category        object
price                    int64
original_price           int64
discount_rate            int64
rating                 float64
review_count             int64
sold_count               int64
origin                  object
is_imported               bool
is_authentic             int64
is_official_store         bool
tiki_verified            int64
has_authentic_badge       bool
availability             int64
dtype: object

--- Tỉ lệ missing (%)
origin    5.3
dtype: float64

--- Thống kê cơ bản
              price  original_price  discount_rate       rating  review_count  \
count  8.141000e+03    8.141000e+03    8141.000000  8141.000000   8141.000000   
mean   5.977270e+05    6.442027e+05       7.418745     1.628547 

## **2. LẤP ĐẦY DỮ LIỆU THIẾU VÀ CHUẨN HÓA CƠ BẢN**

### **2.1. Xử lý giá trị thiếu và trùng lặp**

In [18]:
# 1. origin: điền từ is_imported nếu trống
def fill_origin(row):
    if pd.notna(row["origin"]):
        return row["origin"]
    # Kiểm tra rõ ràng True/False để tránh lỗi nếu is_imported bị Null
    if row["is_imported"] is True:
        return "Ngoài nước (không rõ)"
    if row["is_imported"] is False:
        return "Việt Nam"
    return "Không rõ"

df["origin"] = df.apply(fill_origin, axis=1)

# 2. rating & review_count: sản phẩm mới chưa có đánh giá → điền 0
df["rating"]       = df["rating"].fillna(0)
df["review_count"] = df["review_count"].fillna(0).astype(int)
df["sold_count"]   = df["sold_count"].fillna(0).astype(int)

# 3. brand_name: điền "Không rõ", xử lí lỗi "Viet Nam" → "Việt Nam"
df["brand_name"] = df["brand_name"].fillna("Không rõ")
df["brand_name"] = df["brand_name"].replace({"Viet Nam": "Việt Nam"})

# 4. discount_rate: nếu thiếu thì tính lại từ giá
df["discount_rate"] = df.apply(
    lambda r: r["discount_rate"] if pd.notna(r["discount_rate"]) and r["discount_rate"] > 0
              else (round((r["original_price"] - r["price"]) / r["original_price"] * 100, 1)
                   if r["original_price"] > r["price"] else 0),
    axis=1
)

print("Missing sau xử lý:")
print((df.isnull().mean() * 100).round(1)[lambda x: x > 0])

Missing sau xử lý:
Series([], dtype: float64)


### **2.2. Loại bỏ nhiễu: Dụng cụ & Quà tặng**

In [19]:
# 1. Định nghĩa từ khóa
gift_keywords = ['quà tặng', 'qùa tặng', 'tặng kèm', 'gift', 'không bán', 'bonus', 'tặng', 'theo kèm']
tools_keywords = [
    'cọ', 'mút', 'bông phấn', 'máy rửa mặt', 'máy xông', 'nhíp', 'kẹp mi', 
    'dũa', 'lược', 'túi đựng', 'khay', 'gương', 'giấy thấm dầu', 'bông tẩy trang',
    'máy massage', 'đầu bàn chải', 'hộp đựng', 'bàn chà'
]

# 2. Thực hiện lọc
# Lọc Quà tặng (Dựa trên tên và giá)
filter_gifts = df['name'].str.contains('|'.join(gift_keywords), case=False, na=False) | \
               (df['price'] < 1000) # Thường quà tặng giá sẽ là 0đ hoặc vài trăm đồng

# Lọc Dụng cụ (Dựa trên tên và Category)
filter_tools = df['name'].str.contains('|'.join(tools_keywords), case=False, na=False) | \
               (df['category'] == 'Dụng cụ trang điểm')

df = df[~(filter_gifts | filter_tools)].copy()

# 3. Kiểm tra kết quả
print(f"Số lượng sản phẩm sau khi dọn rác: {len(df)}")

Số lượng sản phẩm sau khi dọn rác: 7266


### **2.3. Chuẩn hóa chuỗi ký tự**

In [20]:
# 1. Định nghĩa từ điển thay thế trước để tránh lỗi NameError
brand_replacement = {
    "The Cocoon Original Vietnam": "Cocoon",
    "The History Of Whoo": "Whoo",
    "L'Oréal Paris": "L'Oréal",
    "Loreal": "L'Oréal",
    "LOREAL": "L'Oréal",
    "Maybelline New York": "Maybelline",
    "Su:M37": "Sum37",
    "Hada Labo": "Hada Labo",
    "Dermalogica": "Dermalogica"
}

# 2. Làm sạch tên sản phẩm
def clean_name(text):
    if not isinstance(text, str): 
        return ""    
    text = re.sub(r'\[.*?\]', '', text)    
    text = re.sub(r'\d+\s?(g|ml|ML|G|gr|kg|tuýp|viên)', '', text, flags=re.I)   
    text = re.sub(r'["\'“”„«»]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df["name_clean"] = df["name"].apply(clean_name)

# 3. Chuẩn hóa brand_name
df['brand_name'] = df['brand_name'].str.title().str.strip()
df['brand_name'] = df['brand_name'].replace(brand_replacement)

# 4. Khôi phục Brand từ tên sản phẩm (Data Recovery)
known_brands = df[df['brand_name'] != "Không rõ"]['brand_name'].unique()

def recover_brand(row):
    if row['brand_name'] == "Không rõ":
        for b in known_brands:
            # So sánh trên name_clean đã làm sạch để chính xác hơn
            if str(b).lower() in str(row['name_clean']).lower(): 
                return b
    return row['brand_name']

df['brand_name'] = df.apply(recover_brand, axis=1)

# 5. Kiểm tra kết quả trên df
changed = (df["name"] != df["name_clean"]).sum()
print(f"Số tên đã được làm sạch: {changed}")
print("\nVí dụ trước/sau:")
sample = df[df["name"] != df["name_clean"]][["name","name_clean"]].head(5)
for _, row in sample.iterrows():
    print(f"   BEFORE: {row['name'][:80]}")
    print(f"   AFTER : {row['name_clean'][:80]}")
    print()

Số tên đã được làm sạch: 5736

Ví dụ trước/sau:
   BEFORE: Sữa rửa mặt Hada Labo chống lão hóa Premium Cleanser Aging Care 100g
   AFTER : Sữa rửa mặt Hada Labo chống lão hóa Premium Cleanser Aging Care

   BEFORE: Sữa rửa mặt dưỡng trắng cao cấp Hada Labo Premium Cleanser Radiance 100g
   AFTER : Sữa rửa mặt dưỡng trắng cao cấp Hada Labo Premium Cleanser Radiance

   BEFORE: Sữa Rửa Mặt Rosette Làm Giảm Mụn Face Wash Pasta Acne Clear 120G
   AFTER : Sữa Rửa Mặt Rosette Làm Giảm Mụn Face Wash Pasta Acne Clear

   BEFORE: Sữa rửa mặt X-Men 100g Detox/Sáng da/Ngừa mụn/Kiểm soát nhờn/Mát lạnh
   AFTER : Sữa rửa mặt X-Men Detox/Sáng da/Ngừa mụn/Kiểm soát nhờn/Mát lạnh

   BEFORE: Sữa rửa mặt cho nam Oxy sạch dịu nhẹ, kiềm dầu dạng kem Oxy Oil Acne Wash 100g
   AFTER : Sữa rửa mặt cho nam Oxy sạch dịu nhẹ, kiềm dầu dạng kem Oxy Oil Acne Wash



## **3. NHẬN DIỆN VÀ LOẠI BỎ DỮ LIỆU BẤT THƯỜNG**

In [21]:
# 1. Thống kê sơ bộ để phát hiện các giá trị bất thường
print("--- Kiểm tra phân phối Giá")
print(df["price"].describe())
print(f"\nSản phẩm giá > 50 triệu (Nghi vấn): {(df['price'] > 50_000_000).sum()}")
print(f"Sản phẩm giá = 0 (Lỗi dữ liệu): {(df['price'] == 0).sum()}")

print("\n--- Kiểm tra phân phối Tỉ lệ giảm giá (Discount)")
print(df["discount_rate"].describe())
print(f"Sản phẩm giảm giá > 90% (Nghi vấn lỗi): {(df['discount_rate'] > 90).sum()}")

# 2. Loại bỏ các bản ghi không thực tế (Data Cleaning)
df = df[df["price"] > 0]                    # Loại bỏ giá bằng 0 (lỗi hệ thống)
df = df[df["price"] <= 50_000_000]          # Giữ ngưỡng 50tr (phù hợp mỹ phẩm xa xỉ/máy làm đẹp)
df = df[df["discount_rate"].between(0, 90)] # Loại bỏ các mức giảm giá phi thực tế (>90%)

# 3. Đánh dấu các sản phẩm "Siêu ngoại lai" (Extreme Outliers)
# Tạo biến doanh thu ước tính để nhận diện các sản phẩm có sức ảnh hưởng quá lớn
df["estimated_revenue"] = df["price"] * df["sold_count"]

# Sử dụng phương pháp IQR với hệ số k=20 để chỉ lọc các ca thực sự đột biến
Q3 = df['estimated_revenue'].quantile(0.75)
Q1 = df['estimated_revenue'].quantile(0.25)
IQR = Q3 - Q1
revenue_threshold = Q3 + 20 * IQR 

# Đánh Flag (Cờ hiệu) thay vì xóa để bảo toàn dữ liệu cho các phân tích đặc thù
df['is_extreme_outlier'] = (df['estimated_revenue'] > revenue_threshold) | (df['sold_count'] > 100000)

# 4. Xử lý mâu thuẫn logic: Sản phẩm ẩn lượt bán (Sold Hidden)
# Một số sản phẩm có đánh giá (rating) nhưng sold_count = 0 (do Tiki ẩn dữ liệu bán)
# Đánh Flag để loại trừ khi thực hiện phân tích tương quan giữa Lượt bán và Đánh giá
df["sold_hidden_flag"] = (df["sold_count"] == 0) & (df["review_count"] > 0)

print(f"\nSản phẩm ẩn lượt bán (sold=0 nhưng có review): {df['sold_hidden_flag'].sum()}")
print(df[df["sold_hidden_flag"]][
    ["name_clean","rating","review_count","sold_count","is_imported"]
].to_string())

# 5. Nhận diện nhóm người bán có doanh số đột biến (Extreme Sellers)
# Các sản phẩm có lượt bán > 100,000 thường là hàng tiêu dùng nhanh, dễ làm lệch biểu đồ
EXTREME_SOLD_THRESHOLD = 100_000
df["is_extreme_seller"] = df["sold_count"] > EXTREME_SOLD_THRESHOLD

print(f"\nExtreme sellers (sold > {EXTREME_SOLD_THRESHOLD:,}): {df['is_extreme_seller'].sum()}")
print(df[df["is_extreme_seller"]][
    ["name_clean","brand_name","sold_count","price","is_imported"]
].sort_values("sold_count", ascending=False).to_string())

# 6. Khắc phục Bug logic: Trạng thái Gian hàng chính hãng (Official Store)
# Thực tế API Tiki thường trả về is_official_store = False cho toàn bộ sản phẩm
# Tinh chỉnh lại logic: Xác định Official Store dựa trên tên Seller hoặc Badge Authentic
official_keywords = ["official store", "chính hãng", "tiki trading"]
df["is_official_store"] = df.apply(
    lambda r: True if (any(kw in str(r["seller_name"]).lower() for kw in official_keywords) 
                       or r["has_authentic_badge"] == True) else False, 
    axis=1
)

# Thống kê kết quả sau khi khắc phục lỗi logic
print(f"\nSau khi hiệu chỉnh trạng thái Official Store:")
print(f"  Số lượng True (Chính hãng): {df['is_official_store'].sum()}")
print(f"  Số lượng False (Thường): {(~df['is_official_store']).sum()}")
print("\nDanh sách 10 nhà bán hàng lớn nhất được gán nhãn Chính hãng:")
print(df[df["is_official_store"]]["seller_name"].value_counts().head(10).to_string())

print(f"\nTổng quy mô dữ liệu sau xử lý ngoại lai: {len(df):,} sản phẩm")

--- Kiểm tra phân phối Giá
count    7.266000e+03
mean     6.258323e+05
std      9.672417e+05
min      1.000000e+04
25%      1.750000e+05
50%      3.150000e+05
75%      6.500000e+05
max      1.800000e+07
Name: price, dtype: float64

Sản phẩm giá > 50 triệu (Nghi vấn): 0
Sản phẩm giá = 0 (Lỗi dữ liệu): 0

--- Kiểm tra phân phối Tỉ lệ giảm giá (Discount)
count    7266.000000
mean        7.553289
std        14.401879
min         0.000000
25%         0.000000
50%         0.000000
75%         7.000000
max        76.000000
Name: discount_rate, dtype: float64
Sản phẩm giảm giá > 90% (Nghi vấn lỗi): 0

Sản phẩm ẩn lượt bán (sold=0 nhưng có review): 14
                                                                                                            name_clean  rating  review_count  sold_count  is_imported
1839  Serum Vàng Dr Natural Astragrace Gold Placenta Premium | Giúp Trắng Da, Ngăn Ngừa Lão Hóa - Nhập Khẩu Chính Hãng     5.0             1           0        False
3981             

## **4. CHUẨN HÓA XUẤT XỨ VÀ PHÂN LOẠI THỊ TRƯỜNG**

In [22]:
# 1. Chuẩn hóa các tên gọi khác nhau về tên chuẩn
ORIGIN_MAP = {
    # Việt Nam
    "việt nam": "Việt Nam", "viet nam": "Việt Nam", "Viet Nam": "Việt Nam", "VIET NAM": "Việt Nam",
    "china / vietnam": "Việt Nam", "việt nam / đài loan": "Việt Nam",
    "nhật bản / việt nam": "Việt Nam", "anh, việt nam": "Việt Nam",

    # Hàn Quốc
    "hàn quốc": "Hàn Quốc", "korea": "Hàn Quốc",
    "hàn quốc / trung quốc": "Hàn Quốc",

    # Nhật Bản
    "nhật bản": "Nhật Bản", "japan": "Nhật Bản",

    # Trung Quốc
    "trung quốc": "Trung Quốc", "china": "Trung Quốc",
    "hk/tq": "Trung Quốc",

    # Các nước khác
    "mỹ": "Mỹ", "usa": "Mỹ",
    "pháp": "Pháp", "france": "Pháp",
    "thái lan": "Thái Lan", "thailand": "Thái Lan",
    "đức": "Đức", "germany": "Đức",
    "anh": "Anh", "uk": "Anh",
    "đài loan": "Đài Loan", "taiwan": "Đài Loan",
    "hong kong": "Hồng Kông", "hk": "Hồng Kông",
    "indonesia": "Indonesia",
    "ấn độ": "Ấn Độ", "india": "Ấn Độ",
    "úc": "Úc", "australia": "Úc",
    "italia": "Ý", "italy": "Ý",
    "tây ban nha": "Tây Ban Nha",
    "p.r.c": "Trung Quốc", "prc": "Trung Quốc",
    "thụy sỹ": "Thụy Sỹ", "switzerland": "Thụy Sỹ",
    "đài loan": "Đài Loan", "taiwan": "Đài Loan",
    "belgium": "Bỉ", "bỉ": "Bỉ",
    "canada": "Canada",
}

VIETNAM_SET = {"Việt Nam"}

def normalize_origin(raw):
    if pd.isna(raw):
        return "Không rõ"
    # Chuẩn hóa unicode NFC để tránh lỗi so sánh chuỗi tiếng Việt
    raw = unicodedata.normalize('NFC', str(raw))
    raw_lower = raw.lower().strip()
    for key, val in ORIGIN_MAP.items():
        if key in raw_lower:
            return val
    return raw.strip()  # giữ nguyên nếu không match

def classify_origin(origin_norm):
    if origin_norm in VIETNAM_SET:
        return "Trong nước"
    if "Việt Nam" in str(origin_norm):   # đa quốc gia có VN cũng tính là trong nước
        return "Trong nước"
    if origin_norm in ("Không rõ", "Không xác định"):  # Gộp trường "Không rõ"
        return "Không rõ"
    return "Ngoài nước"

df["origin_normalized"] = df["origin"].apply(normalize_origin)
df["origin_class"]      = df["origin_normalized"].apply(classify_origin)

# 2. Dò tìm xuất xứ cho các sản phẩm "Không rõ"
def fix_unknown_origin(row):
    if row["origin_class"] == "Không rõ":
        # Ví dụ: nếu tên có chữ "Nhật" hoặc brand Nhật thì gán Ngoài nước
        if any(kw in row['name'].lower() for kw in ['nhật', 'hàn', 'korea', 'japan', 'trung quốc', 'china', 'đài loan', 'taiwan']):
            return "Ngoài nước"
        # Mặc định còn lại có thể gán theo số đông hoặc để "Ngoài nước" vì Tiki thường cào thiếu hàng ngoại
        return "Ngoài nước" 
    return row["origin_class"]

df["origin_class"] = df.apply(fix_unknown_origin, axis=1)

# Kiểm tra
print("=== Xuất xứ sau chuẩn hóa ===")
print(df["origin_normalized"].value_counts().head(20))
print(f"\n=== Phân loại trong/ngoài nước ===")
print(df["origin_class"].value_counts())
print(f"\nTỉ lệ: {df['origin_class'].value_counts(normalize=True).mul(100).round(1).to_string()}")

=== Xuất xứ sau chuẩn hóa ===
origin_normalized
Việt Nam       2340
Hàn Quốc       1530
Nhật Bản       1109
Pháp            572
Mỹ              349
Thái Lan        190
Ý               186
Đài Loan        175
Trung Quốc      128
Úc              112
Tây Ban Nha      85
Đức              84
Anh              68
Nga              61
Canada           50
Ba Lan           44
Hồng Kông        35
Dubai            22
Thổ Nhĩ Kỳ       22
Ấn Độ            18
Name: count, dtype: int64

=== Phân loại trong/ngoài nước ===
origin_class
Ngoài nước    4926
Trong nước    2340
Name: count, dtype: int64

Tỉ lệ: origin_class
Ngoài nước    67.8
Trong nước    32.2


## **5. XÂY DỰNG CÁC CHỈ SỐ PHÂN TÍCH CHIỀU SÂU**

In [23]:
# 1. Phân khúc giá
def price_segment(price):
    if price < 100_000:    return "Dưới 100k"
    if price < 300_000:    return "100k – 300k"
    if price < 700_000:    return "300k – 700k"
    if price < 2_000_000:  return "700k – 2tr"
    return "Trên 2tr"

df["price_segment"] = df["price"].apply(price_segment)

# 2. Mức độ phổ biến dựa trên lượt bán
def popularity_tier(sold):
    if sold == 0:      return "Chưa có lượt bán"
    if sold < 50:      return "Mới"
    if sold < 500:     return "Phổ biến"
    if sold < 2000:    return "Bán chạy"
    return "Bestseller"

df["popularity_tier"] = df["sold_count"].apply(popularity_tier)

# 3. Mức độ đánh giá
def rating_tier(r):
    if r == 0:    return "Chưa có đánh giá"
    if r < 3.5:   return "Thấp (< 3.5)"
    if r < 4.5:   return "Trung bình (3.5–4.5)"
    return "Cao (≥ 4.5)"

df["rating_tier"] = df["rating"].apply(rating_tier)

# 4. Có giảm giá hay không
df["has_discount"] = df["discount_rate"] > 0

# 5. Tạo cột Product_Type (Phân loại nhóm lớn)
type_map = {
    # 1. Skincare (Chăm sóc da mặt)
    "Sữa rửa mặt": "Skincare", "Tẩy trang": "Skincare", "Toner": "Skincare", 
    "Serum": "Skincare", "Kem dưỡng da": "Skincare", "Mặt nạ": "Skincare",
    "Kem chống nắng (mặt)": "Skincare", "Trị mụn & sẹo": "Skincare", 
    "Chống lão hóa": "Skincare", "Tẩy tế bào chết mặt": "Skincare", "Xịt khoáng": "Skincare",
    "Dưỡng mắt": "Skincare", "Bông tẩy trang": "Skincare", "Giấy thấm dầu": "Skincare",
    "Máy rửa mặt": "Skincare", "Máy hút mụn": "Skincare",

    # 2. Makeup & Nail (Trang điểm & Móng)
    "Son môi": "Makeup", "Trang điểm mặt": "Makeup", "Trang điểm mắt": "Makeup",
    "Dụng cụ trang điểm": "Makeup", "Bộ trang điểm": "Makeup", "Trang điểm khác": "Makeup",
    "Chăm sóc móng": "Makeup", "Dụng cụ làm đẹp khác": "Makeup",

    # 3. Body & Personal Care (Chăm sóc cơ thể & Vệ sinh)
    "Sữa tắm": "Body Care", "Dưỡng thể": "Body Care", "Kem chống nắng (thể)": "Body Care",
    "Tẩy tế bào chết body": "Body Care", "Bộ chăm sóc toàn thân": "Body Care",
    "Sản phẩm sạch khuẩn": "Body Care", "Massage toàn thân": "Body Care",
    "Sản phẩm tẩy lông": "Body Care", "Lăn, xịt khử mùi": "Body Care",
    "Dung dịch vệ sinh": "Body Care", "Răng miệng": "Body Care",
    "Bàn chải": "Body Care", "Kem đánh răng": "Body Care", "Tẩy trắng răng": "Body Care",

    # 4. Hair Care (Chăm sóc tóc)
    "Dầu gội & xả": "Hair Care", "Dưỡng tóc": "Hair Care", "Tạo kiểu tóc": "Hair Care",
    "Thuốc nhuộm tóc": "Hair Care", "Bộ chăm sóc tóc": "Hair Care",

    # 5. Fragrance (Nước hoa)
    "Nước hoa nữ": "Fragrance", "Nước hoa nam": "Fragrance", 
    "Xịt thơm cơ thể (Body Mist)": "Fragrance",

    # 6. Supplements (Thực phẩm chức năng làm đẹp)
    "TPCN làm đẹp": "Supplements", "Vitamin": "Supplements", "Thải độc": "Supplements"
}

df["product_type"] = df["category"].map(type_map).fillna("Khác")

# 6. Tính tỉ lệ Review_Ratio
# Công thức: Review_Ratio = Review_Count/Sold_Count
# Nếu Sold_Count = 0 thì tỉ lệ = 0 để tránh lỗi chia cho 0
df["review_ratio"] = df.apply(
    lambda r: round(r["review_count"] / r["sold_count"], 4) if r["sold_count"] > 0 else 0, 
    axis=1
)

# XỬ lí: Flag review_ratio > 1 (review_count > sold_count)
# review_ratio > 1 = số review nhiều hơn lượt bán ghi nhận
# Hợp lệ về mặt nghiệp vụ (Tiki đôi khi ẩn sold_count thực)
# → Đánh flag, không xóa, không dùng khi phân tích tương quan
df["review_ratio_flag"] = df["review_ratio"] > 1
print(f"Số dòng review_ratio > 1: {df['review_ratio_flag'].sum()}")
print("\nChi tiết:")
print(df[df["review_ratio_flag"]][
    ["name_clean","review_count","sold_count","review_ratio"]
].to_string())


# 7. Decode availability thành nhãn đọc được ────────────────
availability_map = {
    1: "Còn hàng",
    3: "Hết hàng tạm thời",
    4: "Ngừng kinh doanh",
    5: "Đặt trước"
}
df["availability_label"] = df["availability"].map(availability_map).fillna("Không rõ")

# 8. Xử lí flag chứa tên trùng lặp (biến thể)
df["has_name_duplicate"] = df["name_clean"].duplicated(keep=False)
print(f"Tên trùng (biến thể): {df['has_name_duplicate'].sum()}")

print("=== Phân khúc giá ===")
print(df["price_segment"].value_counts())
print("\n=== Mức độ phổ biến ===")
print(df["popularity_tier"].value_counts())
print("\n=== Đánh giá ===")
print(df["rating_tier"].value_counts())
print("=== Phân loại nhóm hàng (Product Type) ===")
print(df["product_type"].value_counts())
print("\n=== Tỉ lệ Review trung bình theo nhóm ===")
print(df.groupby("product_type")["review_ratio"].mean())
print("\n=== Phân bố availability_label ===")
print(df["availability_label"].value_counts())

Số dòng review_ratio > 1: 10

Chi tiết:
                                                                                                           name_clean  review_count  sold_count  review_ratio
2182                                                            Kem Nám Trắng Da Tàn Nhang Đồi Mồi (,) Sắc Tiên Today             3           1        3.0000
3789                                           Tẩy Tế Bào Da Chết Natureine Aqua Peel Moisture Peeling Gel - Nhật Bản             4           1        4.0000
4257                         Kem Nền Kết Cấu Dạng Serum Lì Mịn Như Nhung Daisy Doll Nhật Bản BB Serum SPF 30 Mỏng Nhẹ             2           1        2.0000
6952                Nước hoa nam thơm lâu, Charme Sport , nam tính, năng động đầy cuốn hút, đúng chất quý ông            19           8        2.3750
6965                                                               Nước Hoa Nam Charme Guilty thơm nhẹ nhàng nam tính             3           2        1.5000
6986        

## **6. TỔNG KẾT VÀ XUẤT DỮ LIỆU SẠCH**

In [24]:
# 1. Sắp xếp lại cột
cols_order = [
    # 1. Nhóm định danh
    "product_id", "name", "name_clean", "brand_name", "seller_name",
    
    # 2. Nhóm Phân loại & Xuất xứ
    "product_type", "category", "primary_category", 
    "origin_class", "origin_normalized", "is_imported",
    
    # 3. Nhóm Tài chính & Giá cả
    "price", "original_price", "discount_rate", "has_discount", "price_segment",
    
    # 4. Nhóm Hiệu quả kinh doanh & Thị hiếu
    "sold_count", "popularity_tier", "is_extreme_seller", "review_count", "review_ratio", 
    "review_ratio_flag", "rating", "rating_tier", "estimated_revenue",

    # 5. Nhóm Chỉ số niềm tin & Trạng thái
    "is_official_store", "is_authentic", "has_authentic_badge", "tiki_verified", "availability", "availability_label",
    
    # 6. Nhóm Flag & Cờ hiệu
    "sold_hidden_flag", "has_name_duplicate"
]

# 2. Chuyển các cột định danh sang string và các cột phân loại sang category
df['product_id'] = df['product_id'].astype(str)

cat_cols = ['product_type', 'origin_class', 'price_segment', 'popularity_tier', 'rating_tier']
for col in cat_cols:
    df[col] = df[col].astype('category')

# 3. Lọc và sắp xếp lại cột theo danh sách cols_order
# Dùng list comprehension để tránh lỗi nếu một cột nào đó không tồn tại
df_final = df[[c for c in cols_order if c in df.columns]]

# 4. Lưu file từ df_final
path = "../data"
df_final.to_csv(f"{path}/tiki_cosmetics_processed.csv", index=False, encoding="utf-8-sig")

print(f"=== ĐÃ LƯU tiki_cosmetics_processed.csv ===")
print(f"Shape: {df_final.shape}")
print(f"\n--- Preview")
print(df_final[["name_clean","origin_class","price_segment","rating","sold_count"]].head(8).to_string())

=== ĐÃ LƯU tiki_cosmetics_processed.csv ===
Shape: (7266, 33)

--- Preview
                                                                        name_clean origin_class price_segment  rating  sold_count
1                  Sữa rửa mặt Hada Labo chống lão hóa Premium Cleanser Aging Care   Ngoài nước   100k – 300k     0.0          31
2              Sữa rửa mặt dưỡng trắng cao cấp Hada Labo Premium Cleanser Radiance   Ngoài nước   100k – 300k     5.0          19
3                      Sữa Rửa Mặt Rosette Làm Giảm Mụn Face Wash Pasta Acne Clear   Ngoài nước   100k – 300k     5.0           6
4                 Sữa rửa mặt X-Men Detox/Sáng da/Ngừa mụn/Kiểm soát nhờn/Mát lạnh   Trong nước     Dưới 100k     5.0          18
5        Sữa rửa mặt cho nam Oxy sạch dịu nhẹ, kiềm dầu dạng kem Oxy Oil Acne Wash   Trong nước     Dưới 100k     5.0          56
6  Sữa rửa mặt cho nam Oxy sạch dịu nhẹ, giảm khô căng dạng gel Oxy Hydrating Wash   Trong nước     Dưới 100k     4.3          49
7              